# ML-09 — Validation and Research Claim Audit

This notebook reviews two observational findings from the research paper, compares row-random and client-held-out evaluation on the bundled anonymized snapshot, checks for label and time-window leakage, inspects held-out errors, and narrows the model claim to what the evidence supports.

The March warehouse experiment is separately documented in Week 5; this local audit is explicit about the difference in data and label.

## 1. Two paper findings + my methodology questions

I read *The State of AI-Driven SEO in Numbers* (FlyRank, March 2026) and chose two findings as examples for constructive methodology questions.

**Finding 1 — “The Anatomy of Growing Content” (paper p. 6).** The paper reports that the rising-impressions group averaged about 3,180 words and 184 days old, versus about 2,311 words and 230 days for the falling-impressions group. The paper defines trend direction from the 30-day vs previous-30-day impression change, with a decline below -10%. **Method question:** Could we clarify whether the word-count and age comparisons are strictly descriptive of those same contemporaneous groups, or whether any later outcome validates the pattern? Because the trend label is defined from recent impressions, this finding supports an observed association, not a forecast that longer or younger content will grow.

**Finding 2 — “The Freshness Multiplier” (paper p. 9).** The paper reports a 7.88:1 growth-to-decline ratio for the 31–90-day freshness band and explicitly cautions that the 361+ day band’s 283:1 ratio is unstable because it has only one declining page. **Method question:** Could we see the counts and the ratio re-computed within brands, or with uncertainty intervals, for the 31–90-day result? Also, since growth/decline uses the 30-day trend label and pages selected for a recent refresh may differ from untouched pages, what validation would support “refresh timing” as more than an association? A matched comparison or prospective holdout could help distinguish selection from change after refresh.

These questions respect the paper’s disclosed observational scope; they ask what the design can support, not whether the work is useful.

In [1]:
paper_findings = [
    ("Finding 1, paper p. 6", "Growing vs declining groups differed in mean word count and age.", "The trend label is a contemporaneous 30-day impression change; ask whether the pattern is prospective or descriptive."),
    ("Finding 2, paper p. 9", "The 31-90 day freshness band had a reported 7.88:1 growth-to-decline ratio.", "Ask for within-brand robustness and validation beyond the trend-derived groups; the 361+ tail is explicitly sparse."),
]
for finding, evidence, methodology_question in paper_findings:
    print(f"{finding}: {evidence}\n  Methodology question: {methodology_question}")

Finding 1, paper p. 6: Growing vs declining groups differed in mean word count and age.
  Methodology question: The trend label is a contemporaneous 30-day impression change; ask whether the pattern is prospective or descriptive.
Finding 2, paper p. 9: The 31-90 day freshness band had a reported 7.88:1 growth-to-decline ratio.
  Methodology question: Ask for within-brand robustness and validation beyond the trend-derived groups; the 361+ tail is explicitly sparse.


## 2. My model under an honest split (before/after)

The Week-5 notebook uses logistic regression on the gated March warehouse: first-half features predict a second-half decline proxy, with a client-held-out split. That time-indexed table is not bundled with this repo. To keep this notebook runnable without a warehouse token, I re-use its logistic-regression evaluation pattern and five search-signal families on the bundled anonymized snapshot, comparing a stratified row-random split (before) with a client-held-out split (after).

**Scope limit:** this is a local snapshot audit, not a byte-for-byte rerun of the March warehouse experiment. The local label is `trend_direction == “down”`, and its 90-day features overlap the trend window. The grouped split tests transfer to unseen clients; it does not remove that temporal overlap. Treat all reported metrics as observed label agreement, not future-period forecasting.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd()
while not (ROOT / "data" / "raw" / "content_refresh_anonymized.csv").is_file():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Could not locate the bundled anonymized CSV from the notebook directory.")
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
data = pd.read_csv(DATA_PATH)
required_columns = {
    "content_id", "client_id", "trend_direction", "trend_pct", "impressions_90d",
    "clicks_90d", "avg_position", "days_with_impressions", "ctr", "content_age_days",
}
missing_columns = sorted(required_columns - set(data.columns))
if missing_columns:
    raise ValueError(f"The bundled dataset is missing required audit columns: {missing_columns}")

for column in ["impressions_90d", "clicks_90d", "avg_position", "days_with_impressions", "ctr", "content_age_days", "trend_pct"]:
    data[column] = pd.to_numeric(data[column], errors="coerce").replace([np.inf, -np.inf], np.nan)
data["trend_direction"] = data["trend_direction"].fillna("unknown").astype(str).str.lower()
data = data.loc[(data["impressions_90d"] > 0) & (data["content_age_days"] >= 90)].copy()
data = data.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
data["target"] = data["trend_direction"].eq("down").astype(int)

FEATURE_COLUMNS = [
    "impressions_90d", "clicks_90d", "avg_position", "days_with_impressions", "ctr",
]
X = data[FEATURE_COLUMNS].copy()
y = data["target"]
groups = data["client_id"].fillna("unknown").astype(str)
if y.nunique() != 2 or groups.nunique() < 5:
    raise ValueError("The eligible data must have both target classes and at least five client groups.")

row_train_idx, row_test_idx = next(
    StratifiedShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(X, y)
)
group_train_idx, group_test_idx = next(
    GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(X, y, groups=groups)
)
if set(groups.iloc[group_train_idx]).intersection(groups.iloc[group_test_idx]):
    raise AssertionError("Client IDs crossed the grouped train/test boundary.")
for train_idx, test_idx in [(row_train_idx, row_test_idx), (group_train_idx, group_test_idx)]:
    if y.iloc[train_idx].nunique() != 2 or y.iloc[test_idx].nunique() != 2:
        raise ValueError("Both target classes must be present in each train and test split.")

def make_model():
    return make_pipeline(
        SimpleImputer(strategy="median", add_indicator=True),
        StandardScaler(),
        LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    )

def precision_at_k(labels, scores, k):
    labels = np.asarray(labels)
    scores = np.asarray(scores)
    top_positions = np.argsort(-scores, kind="stable")[: min(k, len(labels))]
    return float(labels[top_positions].mean())

results = []
group_test_scores = None
for split_name, train_idx, test_idx in [
    ("Before: stratified row-random", row_train_idx, row_test_idx),
    ("After: client-held-out", group_train_idx, group_test_idx),
]:
    fitted_model = make_model()
    fitted_model.fit(X.iloc[train_idx], y.iloc[train_idx])
    test_scores = fitted_model.predict_proba(X.iloc[test_idx])[:, 1]
    test_labels = y.iloc[test_idx].to_numpy()
    results.append({
        "split": split_name,
        "test_rows": len(test_idx),
        "test_clients": groups.iloc[test_idx].nunique(),
        "base_rate": float(test_labels.mean()),
        "ROC-AUC": float(roc_auc_score(test_labels, test_scores)),
        "average_precision": float(average_precision_score(test_labels, test_scores)),
        "precision@50": precision_at_k(test_labels, test_scores, 50),
        "precision@100": precision_at_k(test_labels, test_scores, 100),
    })
    if split_name == "After: client-held-out":
        group_test_scores = test_scores

comparison = pd.DataFrame(results)
print(f"Eligible rows: {len(data):,}; distinct clients: {groups.nunique():,}; overall decline-label rate: {y.mean():.3f}")
print("Precision@K and average precision are shown next to each split's base rate.")
display(comparison.round(3))
print(f"Client overlap in grouped split: {len(set(groups.iloc[group_train_idx]).intersection(groups.iloc[group_test_idx]))}")

group_test = X.iloc[group_test_idx].reset_index(drop=True).copy()
group_test["observed_label"] = y.iloc[group_test_idx].to_numpy()
group_test["score"] = group_test_scores
group_test["predicted_label"] = (group_test_scores >= 0.5).astype(int)
group_test["error_type"] = np.select(
    [(group_test["predicted_label"] == 1) & (group_test["observed_label"] == 0),
     (group_test["predicted_label"] == 0) & (group_test["observed_label"] == 1)],
    ["false positive", "false negative"],
    default="correct",
)
group_error_counts = group_test["error_type"].value_counts()
false_positive_count = int(group_error_counts.get("false positive", 0))
false_negative_count = int(group_error_counts.get("false negative", 0))
print(f"Grouped-test false positives: {false_positive_count:,}; false negatives: {false_negative_count:,}")
if false_positive_count == 0 or false_negative_count == 0:
    raise ValueError("The grouped holdout must contain both false-positive and false-negative examples.")

error_examples = pd.DataFrame([
    {
        "observed_error_pattern": "False positive: a held-out record ranked above the cutoff without the decline-proxy label.",
        "human_review_question": "Could short-window volume variation, seasonality, or a measurement change explain the mismatch?",
    },
    {
        "observed_error_pattern": "False negative: a held-out record met the decline-proxy label but ranked below the cutoff.",
        "human_review_question": "Did a signal outside the measured feature window precede the observed change?",
    },
])
print("These are real grouped-holdout error patterns; record-level feature values are withheld from the public notebook.")
display(error_examples)

Eligible rows: 30,000; distinct clients: 32; overall decline-label rate: 0.542
Precision@K and average precision are shown next to each split's base rate.


,split,test_rows,test_clients,base_rate,ROC-AUC,average_precision,precision@50,precision@100
0,Before: stratified row-random,7500,31,0.542,0.620,0.635,0.84,0.77
1,After: client-held-out,7115,8,0.517,0.562,0.575,0.84,0.78


Client overlap in grouped split: 0
Grouped-test false positives: 1,675; false negatives: 1,670
These are real grouped-holdout error patterns; record-level feature values are withheld from the public notebook.


,observed_error_pattern,human_review_question
0,False positive: a held-out record ranked above...,"Could short-window volume variation, seasonali..."
1,False negative: a held-out record met the decl...,Did a signal outside the measured feature wind...


## 3. Leakage audit

The current label is `trend_direction == “down”`; the data contract identifies `trend_direction` as derived from `trend_pct`. Both are excluded from predictors, along with `content_id` and `client_id` (the latter is used only to form groups). The selected 90-day search metrics do not directly contain the label field, but their rolling window may overlap the same 30-day trend used to make the label. The CSV does not provide a historical feature cutoff that would prove those metrics were available before that outcome window. The client-held-out split fixes repeated-client overlap, not this temporal limitation.

In [3]:
excluded_columns = {
    "target", "is_declining_label", "trend_direction", "trend_pct",
    "content_id", "client_id",
}
if set(FEATURE_COLUMNS).intersection(excluded_columns):
    raise AssertionError("A target, target-derived field, or identifier was included as a model feature.")

feature_audit = pd.DataFrame([
    {"field(s)": "is_declining_label / trend_direction", "decision": "Target only / excluded", "reason": "The model outcome is trend_direction == down."},
    {"field(s)": "trend_pct", "decision": "Excluded; direct label sibling", "reason": "The data contract says trend_direction is derived from trend_pct."},
    {"field(s)": "content_id / client_id", "decision": "Excluded from X", "reason": "Pseudonymous IDs are used only for deduplication and client grouping."},
    {"field(s)": ", ".join(FEATURE_COLUMNS), "decision": "Used, with temporal-overlap caveat", "reason": "These are 90-day snapshot metrics; their availability before the 30-day trend outcome cannot be verified from this CSV."},
    {"field(s)": "Action flags / model scores", "decision": "Excluded", "reason": "Decision-derived fields would reproduce an existing workflow rather than add independent evidence."},
])
display(feature_audit)

leak_smoke_model = make_model()
leak_smoke_X = data[["trend_pct"]]
leak_smoke_model.fit(leak_smoke_X.iloc[group_train_idx], y.iloc[group_train_idx])
leak_smoke_scores = leak_smoke_model.predict_proba(leak_smoke_X.iloc[group_test_idx])[:, 1]
print("Intentional leakage smoke test only — trend_pct is a label sibling and is NOT retained as a feature.")
print(f"Excluded trend_pct ROC-AUC on the same held-out clients: {roc_auc_score(y.iloc[group_test_idx], leak_smoke_scores):.3f}")
print(f"Selected-feature client-held-out ROC-AUC: {comparison.loc[comparison['split'] == 'After: client-held-out', 'ROC-AUC'].iloc[0]:.3f}")

,field(s),decision,reason
0,is_declining_label / trend_direction,Target only / excluded,The model outcome is trend_direction == down.
1,trend_pct,Excluded; direct label sibling,The data contract says trend_direction is deri...
2,content_id / client_id,Excluded from X,Pseudonymous IDs are used only for deduplicati...
3,"impressions_90d, clicks_90d, avg_position, day...","Used, with temporal-overlap caveat",These are 90-day snapshot metrics; their avail...
4,Action flags / model scores,Excluded,Decision-derived fields would reproduce an exi...


Intentional leakage smoke test only — trend_pct is a label sibling and is NOT retained as a feature.
Excluded trend_pct ROC-AUC on the same held-out clients: 1.000
Selected-feature client-held-out ROC-AUC: 0.562


## 4. Claim rewrite

The earlier model report's headline score can sound like broad predictive capability. Here is a narrower rewrite tied to this audit's own measured split. “Observed,” “measured,” and “directional” describe the result; “decision-support” describes the intended use, not an automated decision.

In [4]:
grouped_result = comparison.loc[comparison["split"] == "After: client-held-out"].iloc[0]
safe_claim = (
    f"On the eligible anonymized snapshot, the logistic-regression score measured "
    f"ROC-AUC {grouped_result['ROC-AUC']:.3f} and precision@50 {grouped_result['precision@50']:.3f} "
    f"on one client-held-out split, against a test base rate of {grouped_result['base_rate']:.3f}. "
    "This is a directional decision-support signal for prioritizing human review. It does not establish "
    "future-period performance or that a refresh causes recovery; the rolling feature window may overlap the label window."
)
print("Rewritten claim:")
print(safe_claim)

Rewritten claim:
On the eligible anonymized snapshot, the logistic-regression score measured ROC-AUC 0.562 and precision@50 0.840 on one client-held-out split, against a test base rate of 0.517. This is a directional decision-support signal for prioritizing human review. It does not establish future-period performance or that a refresh causes recovery; the rolling feature window may overlap the label window.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.